In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from datetime import datetime
from pathlib import Path
from plotly.subplots import make_subplots
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# __Previsão de curto prazo do consumo residencial de energia elétrica__
## __Universidade de São Paulo__

### *Disciplina: Deep Learning - MAE5920 - Prof. Dr. Hamed Yazdanpanah*
### *Ian Teixeira Barreiro (10717526) e Ornella Scardua Ferreira (ADD NUSP)*

_Fonte:_ https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction

_Descrição dos Dados:_ The data set is at 10 min for about 4.5 months. The house temperature and humidity conditions were monitored with a ZigBee wireless sensor network. Each wireless node transmitted the temperature and humidity conditions around 3.3 min. Then, the wireless data was averaged for 10 minutes periods. The energy data was logged every 10 minutes with m-bus energy meters. Weather from the nearest airport weather station (Chievres Airport, Belgium) was downloaded from a public data set from Reliable Prognosis (rp5.ru), and merged together with the experimental data sets using the date and time column. Two random variables have been included in the data set for testing the regression models and to filter out non predictive attributes (parameters).

_Descrição da tarefa:_ Prever o consumo de energia dos eletrodomésticos com horizontes de 10 e 60 minutos.

__Modelos sugeridos:__

    - Baselines: regressão linear autorregressiva com variáveis exógenas.

    - Modelo profundo: rede GRU ou rede neural convolucional unidimensional (CNN 1D) de
pequeno porte.

__Avaliação:__

    - Erro absoluto médio (MAE) e raiz do erro quadrático médio (RMSE).

    - Erro absoluto médio escalonado (MASE), utilizando a previsão por persistência como
referência.

    - Desempenho das previsões por hora do dia.

__Entregáveis finais:__

    - Um notebook ou projeto Python reproduzível.

    - Um dicionário de dados resumido e uma análise de possíveis vazamentos de informação
(data leakage).

    - Implementações de modelos de referência (baselines) e de aprendizagem profunda.

    - Uma tabela com os resultados obtidos em dados de treino, validação e teste.

    - Análises de calibração ou diagnósticos dos resíduos.

    - Um resumo do custo computacional.

    - Um relatório de aproximadamente 6 a 8 páginas.

    - Uma apresentação de 10 a 12 minutos.

    - Uma seção final intitulada “Quando o modelo de aprendizagem profunda falha?”.

## Data Loading

In [2]:
ROOT = Path(__name__).resolve().parent.parent
DATAPATH = ROOT / 'data'

original_df = pd.read_csv(f'{DATAPATH}/energydata_complete.csv')

## EDA

In [3]:
original_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         19735 non-null  str    
 1   Appliances   19735 non-null  int64  
 2   lights       19735 non-null  int64  
 3   T1           19735 non-null  float64
 4   RH_1         19735 non-null  float64
 5   T2           19735 non-null  float64
 6   RH_2         19735 non-null  float64
 7   T3           19735 non-null  float64
 8   RH_3         19735 non-null  float64
 9   T4           19735 non-null  float64
 10  RH_4         19735 non-null  float64
 11  T5           19735 non-null  float64
 12  RH_5         19735 non-null  float64
 13  T6           19735 non-null  float64
 14  RH_6         19735 non-null  float64
 15  T7           19735 non-null  float64
 16  RH_7         19735 non-null  float64
 17  T8           19735 non-null  float64
 18  RH_8         19735 non-null  float64
 19  T9           19

In [4]:
original_df.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


Vemos que não há dados faltantes em nenhum das variáveis. Há uma variável de datas que está no formato datetime mas está como str. As demais variáveis são inteiras ou float. Segue uma descrição das variáveis.

__Variables:__

    - Appliances, energy use in Wh
    - lights, energy use of light fixtures in the house in Wh
    - T1, Temperature in kitchen area, in Celsius
    - RH_1, Humidity in kitchen area, in %
    - T2, Temperature in living room area, in Celsius
    - RH_2, Humidity in living room area, in %
    - T3, Temperature in laundry room area
    - RH_3, Humidity in laundry room area, in %
    - T4, Temperature in office room, in Celsius
    - RH_4, Humidity in office room, in %
    - T5, Temperature in bathroom, in Celsius
    - RH_5, Humidity in bathroom, in %
    - T6, Temperature outside the building (north side), in Celsius
    - RH_6, Humidity outside the building (north side), in %
    - T7, Temperature in ironing room , in Celsius
    - RH_7, Humidity in ironing room, in %
    - T8, Temperature in teenager room 2, in Celsius
    - RH_8, Humidity in teenager room 2, in %
    - T9, Temperature in parents room, in Celsius
    - RH_9, Humidity in parents room, in %
    - To, Temperature outside (from Chievres weather station), in Celsius
    - Pressure (from Chievres weather station), in mm Hg
    - RH_out, Humidity outside (from Chievres weather station), in %
    - Wind speed (from Chievres weather station), in m/s
    - Visibility (from Chievres weather station), in km
    - Tdewpoint (from Chievres weather station), Â°C
    - rv1, Random variable 1, nondimensional
    - rv2, Random variable 2, nondimensional

Ou seja, as variáveis indicam consumo de energia elétrica por luzes e eletrodomésticos, humidade e temperatura em cada cômodo, condições climáticas e duas variáveis aleatórias.

Vamos adianter um pequeno passo de preprocessamento para facilitar o EDA, tranformando as datas em datetime.

In [5]:
original_df['date'] = pd.to_datetime(original_df['date'], format='%Y-%m-%d %H:%M:%S')
original_df = original_df.set_index('date', drop=False)

print(original_df['date'].dtype)
print(f"Período: {original_df['date'].min()} até {original_df['date'].max()}")
print(f"Intervalos entre observações: {original_df['date'].diff().value_counts().to_dict()}")

datetime64[us]
Período: 2016-01-11 17:00:00 até 2016-05-27 18:00:00
Intervalos entre observações: {Timedelta('0 days 00:10:00'): 19734}


Usaremos o mapeamento abaixo para dar nomes legíveis aos cômodos nos gráficos. Note que `T6`/`RH_6` são medidos do lado de fora do prédio (lado norte), portanto não entram como cômodos da casa.

In [6]:
ROOMS = {
    '1': 'Cozinha',
    '2': 'Sala de estar',
    '3': 'Lavanderia',
    '4': 'Escritório',
    '5': 'Banheiro',
    '7': 'Sala de passar',
    '8': 'Quarto adolescente',
    '9': 'Quarto dos pais',
}
TEMP_COLS = [f'T{i}' for i in ROOMS]
HUM_COLS = [f'RH_{i}' for i in ROOMS]

original_df[['Appliances', 'lights'] + TEMP_COLS + HUM_COLS + ['T6', 'RH_6', 'T_out', 'RH_out']].describe().T

,count,mean,std,min,25%,50%,75%,max
Appliances,19735.0,97.694958,102.524891,10.000000,50.000000,60.000000,100.000000,1080.000000
lights,19735.0,3.801875,7.935988,0.000000,0.000000,0.000000,0.000000,70.000000
T1,19735.0,21.686571,1.606066,16.790000,20.760000,21.600000,22.600000,26.260000
T2,19735.0,20.341219,2.192974,16.100000,18.790000,20.000000,21.500000,29.856667
T3,19735.0,22.267611,2.006111,17.200000,20.790000,22.100000,23.290000,29.236000
T4,19735.0,20.855335,2.042884,15.100000,19.530000,20.666667,22.100000,26.200000
T5,19735.0,19.592106,1.844623,15.330000,18.277500,19.390000,20.619643,25.795000
T7,19735.0,20.267106,2.109993,15.390000,18.700000,20.033333,21.600000,26.000000
T8,19735.0,22.029107,1.956162,16.306667,20.790000,22.100000,23.390000,27.230000
T9,19735.0,19.485828,2.014712,14.890000,18.000000,19.390000,20.600000,24.500000


Podemos ver que há uma diferença nas médias de temperaturas entre os cômodos. Comparando médias e medianas, a distribuição das temperaturas de todos os cômodos parece ser simétrica. Mas alguns cômodos apresentam desvio-padrão um pouco maior que outros. O mesmo ocorre para humidade.

Para eletrodoméstricos e luzes, vemos significativa assimetria. Também vemos, como esperado, que eletrodomésticos consomem mais energia elétrica que luzes, e com maior variância.

### Consumo de energia ao longo do tempo

A série original tem resolução de 10 minutos e é bastante ruidosa, então plotamos também a média diária para evidenciar a tendência ao longo dos ~4,5 meses (de janeiro a maio).

In [7]:
daily = original_df[['Appliances', 'lights']].resample('D').mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('Eletrodomésticos (Wh)', 'Iluminação (Wh)'))
for row, col in enumerate(['Appliances', 'lights'], start=1):
    fig.add_trace(go.Scattergl(x=original_df.index, y=original_df[col], mode='lines', name=f'{col} (10 min)',
                               line=dict(width=0.5), opacity=0.4), row=row, col=1)
    fig.add_trace(go.Scatter(x=daily.index, y=daily[col], mode='lines', name=f'{col} (média diária)',
                             line=dict(width=2)), row=row, col=1)
fig.update_layout(height=650, title='Consumo de energia ao longo do tempo')
fig.show()

Como podemos ver, em cada dia no grafo de 10 em 10 minutos há bastante oscilação no consumo de energia elétrica, indicando que seria interessante observar o comporamento diário. Visualmente, é possível ver que há dias de maior consumo tanto de energia por eletrodomésticos quanto por luzes. Não é possível detectar nenhuma sazonalidade visualmente por enquanto.

Como a tarefa envolve avaliar o desempenho por hora do dia, vale olhar o perfil médio de consumo por hora e por dia da semana.

In [8]:
profile = original_df[['Appliances', 'lights']].assign(
    hora=original_df.index.hour,
    dia_semana=original_df.index.day_name(),
)

fig = px.line(profile.groupby('hora')[['Appliances', 'lights']].mean(), markers=True,
              title='Consumo médio por hora do dia', labels={'value': 'Wh', 'variable': ''})
fig.show()

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heat = profile.pivot_table(index='dia_semana', columns='hora', values='Appliances', aggfunc='mean').loc[weekday_order]
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Viridis',
                title='Consumo médio dos eletrodomésticos (Wh) por dia da semana e hora',
                labels={'x': 'Hora', 'y': '', 'color': 'Wh'})
fig.show()

Vemos que há baixo consumo de energia de manhã e à noite. Durante a manhã, o consumo vai aumentando. À partir das 16 horas, no final da tarde, o consumo vai aumentando, tendo seu pico às 18 horas. À partir desse horário o consumo vai baixando. Tal comportamento pode ser facilmente lido em termos comportamentais. 

Vemos que às 18 horas, para todos os dias da semana há esse pico, provavelmente correspondendo ao horário de retorno do trabalho e início dos a fazeres domésticos. Pela tarde, na terça-feira, quarta-feira, quinta-feira e domingo há menos consumo de energia por eletrodomésticos, potencialmente por serem dias que as pessoas não estão em casa. Possivelmente, segunda-feira e sexta são dias de home-office, pelo consumo alto pela tarde. 

In [9]:
fig = px.histogram(original_df, x='Appliances', nbins=100, title='Distribuição do consumo dos eletrodomésticos')
fig.show()

Vemos que há dias de altíssimo consumo por appliances, mas esses são outliers.

### Temperatura nos cômodos

Para não sobrecarregar o gráfico com 8 séries de ~20 mil pontos, usamos a média horária. O boxplot ao lado permite comparar a distribuição de temperatura de cada cômodo.

In [10]:
def plot_rooms(cols, prefix, unit, title):
    hourly = (original_df[cols].resample('h').mean()
              .rename(columns=lambda c: ROOMS[c.removeprefix(prefix)]))
    fig = px.line(hourly, title=f'{title} por cômodo (média horária)',
                  labels={'value': unit, 'date': '', 'variable': 'Cômodo'})
    fig.show()

    long = (original_df[cols].rename(columns=lambda c: ROOMS[c.removeprefix(prefix)])
            .melt(var_name='Cômodo', value_name=unit))
    fig = px.box(long, x='Cômodo', y=unit, color='Cômodo', title=f'Distribuição de {title.lower()} por cômodo')
    fig.update_layout(showlegend=False)
    fig.show()


plot_rooms(TEMP_COLS, 'T', '°C', 'Temperatura')

Percebe-se que na média, os cômodos exibem comportamento semelhante em termos de temperatura, como de se esperar pela influência da temperatura externa. No entanto, há cômodos com médias diferentes, e variâncias diferentes, como já haviamos percebido. É interessante perceber a grande quantidade de pontos extremos positivos na sala de estar e na lavanderia, indicando momentos de temperatura atípica nesses cômodos.

### Umidade nos cômodos

In [11]:
plot_rooms(HUM_COLS, 'RH_', '%', 'Umidade')

Banheiro óbviamente bem mais úmido. A sala de passar é mais seca que a média. Demais cômodos possuem distribuição muito parecida. Cozinha tem momentos de alta umidade, como de se esperar dependendo do que se cozinha. Sala de estar tem momentos de baixa umidade. Potencialmente pelo uso de ar condicionado ou ventiladores?

### Condições externas

Comparamos a temperatura e umidade da estação meteorológica de Chièvres (`T_out`, `RH_out`) com as do sensor instalado do lado de fora do prédio (`T6`, `RH_6`).

In [12]:
hourly_out = original_df[['T_out', 'T6', 'RH_out', 'RH_6']].resample('h').mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('Temperatura externa (°C)', 'Umidade externa (%)'))
fig.add_trace(go.Scatter(x=hourly_out.index, y=hourly_out['T_out'], name='T_out (estação)'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_out.index, y=hourly_out['T6'], name='T6 (lado norte)'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_out.index, y=hourly_out['RH_out'], name='RH_out (estação)'), row=2, col=1)
fig.add_trace(go.Scatter(x=hourly_out.index, y=hourly_out['RH_6'], name='RH_6 (lado norte)'), row=2, col=1)
fig.update_layout(height=650, title='Temperatura e umidade externas (média horária)')
fig.show()

Vemos que a temperatura é praticamente idêntica. No entanto, à partir de um certo momento, a umidade na estação é consistentemente maior que a do lado de fora da casa.

### Diferença entre temperatura interna e externa

A temperatura média da casa é a média das temperaturas dos 8 cômodos (soma das temperaturas dividida pelo número de cômodos). A diferença `T_casa - T_out` indica o quanto a casa está mais quente que o ambiente externo.

In [13]:
temp_diff = pd.DataFrame({
    'T_casa': original_df[TEMP_COLS].mean(axis=1),
    'T_out': original_df['T_out'],
})
temp_diff['T_casa - T_out'] = temp_diff['T_casa'] - temp_diff['T_out']
hourly_diff = temp_diff.resample('h').mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('Temperatura média da casa vs. externa (°C)', 'Diferença T_casa - T_out (°C)'))
fig.add_trace(go.Scatter(x=hourly_diff.index, y=hourly_diff['T_casa'], name='T_casa'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_diff.index, y=hourly_diff['T_out'], name='T_out'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_diff.index, y=hourly_diff['T_casa - T_out'], name='T_casa - T_out',
                         fill='tozeroy'), row=2, col=1)
fig.update_layout(height=650, title='Temperatura interna vs. externa (média horária)')
fig.show()

Aqui devemos desconsiderar a diferença na variância, já que ela foi controlada pela média que tiramos dos cômodos. O ponto é perceber que de modo geral a casa está mais quente que o ambiente externo.

### Correlação entre as variáveis

Por fim, a matriz de correlação dá uma primeira ideia de quais variáveis se relacionam com o consumo dos eletrodomésticos. As variáveis aleatórias `rv1` e `rv2` servem de referência para correlações espúrias.

In [14]:
corr = original_df.drop(columns='date').corr()
fig = px.imshow(corr, color_continuous_scale='RdBu_r', zmin=-1, zmax=1, aspect='auto',
                title='Matriz de correlação de Pearson')
fig.update_layout(height=800)
fig.show()

Temperaturas tem forte correlação positiva entre sí e umidades também. A umidade do lado de fora da casa (RH6) está negativamente correlacionada com as temperaturas dentro de cada - se chove, a temperatura dentro de casa abaixa. O mesmo em um grau menor com a umidade da estação de observação. 

## Preprocessing

Nesta seção construímos os conjuntos de treino, validação e teste **comuns** a todos os modelos, para que os resultados sejam comparáveis.

__Variáveis resposta.__ A tarefa pede a previsão do consumo dos eletrodomésticos (`Appliances`, renomeada para `appliances`) com horizontes de 10 e 60 minutos. Como os dados têm resolução de 10 minutos, as respostas são o próprio `appliances` deslocado 1 e 6 passos à frente:

- `y_10` $= y_{t+1}$, consumo daqui a 10 minutos;
- `y_60` $= y_{t+6}$, consumo daqui a 60 minutos.

Em cada linha, as variáveis explicativas são as observadas no instante $t$ (incluindo o próprio `appliances`, que é a parte autorregressiva) e as respostas são valores futuros.

__Variáveis explicativas.__ Renomeamos as colunas para nomes mais intuitivos (mapeamento completo em `COLUMN_NAMES`): `<cômodo>_temp` e `<cômodo>_hum` para os cômodos, `outside_north_temp`/`outside_north_hum` para o sensor do lado de fora do prédio (`T6`/`RH_6`) e `weather_*` para a estação meteorológica de Chièvres. Removemos `rv1` e `rv2`, que são ruído por construção (foram incluídas no dataset original justamente para testar seleção de variáveis). Como a EDA mostrou um padrão diário forte e diferenças entre os dias da semana, adicionamos:

- `hour_sin`, `hour_cos`: hora do dia em codificação cíclica (23h50 fica próximo de 00h00);
- `dow_0`, ..., `dow_6`: dia da semana em one-hot (0 = segunda-feira).

In [ ]:
HORIZONS = {'y_10': 1, 'y_60': 6}  # horizonte em passos de 10 minutos
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # teste = restante

COLUMN_NAMES = {
    'Appliances': 'appliances',
    'T1': 'kitchen_temp', 'RH_1': 'kitchen_hum',
    'T2': 'living_room_temp', 'RH_2': 'living_room_hum',
    'T3': 'laundry_temp', 'RH_3': 'laundry_hum',
    'T4': 'office_temp', 'RH_4': 'office_hum',
    'T5': 'bathroom_temp', 'RH_5': 'bathroom_hum',
    'T6': 'outside_north_temp', 'RH_6': 'outside_north_hum',
    'T7': 'ironing_room_temp', 'RH_7': 'ironing_room_hum',
    'T8': 'teen_room_temp', 'RH_8': 'teen_room_hum',
    'T9': 'parents_room_temp', 'RH_9': 'parents_room_hum',
    'T_out': 'weather_temp', 'RH_out': 'weather_hum',
    'Press_mm_hg': 'weather_pressure', 'Windspeed': 'weather_windspeed',
    'Visibility': 'weather_visibility', 'Tdewpoint': 'weather_dewpoint',
}

df = original_df.drop(columns=['date', 'rv1', 'rv2']).rename(columns=COLUMN_NAMES)

minutes = df.index.hour * 60 + df.index.minute
df['hour_sin'] = np.sin(2 * np.pi * minutes / 1440)
df['hour_cos'] = np.cos(2 * np.pi * minutes / 1440)
df = df.join(pd.get_dummies(df.index.dayofweek, prefix='dow', dtype=int).set_index(df.index))

df.columns.tolist()

### Divisão em treino, validação e teste

A divisão é __temporal__ (sem embaralhamento): treinamos no passado e avaliamos no futuro, como aconteceria em produção. Embaralhar as linhas vazaria informação, já que observações vizinhas de 10 em 10 minutos são quase idênticas.

Usamos 70% / 15% / 15%, com os cortes arredondados para a meia-noite, de forma que cada conjunto contenha apenas dias completos. Isso dá ~95 dias de treino e ~3 semanas para validação e para teste. Preferimos 15% a 10% na validação porque 10% seriam só 2 semanas, e o consumo médio semanal varia bastante (de ~75 a ~120 Wh), o que deixaria a escolha de hiperparâmetros e o early stopping sensíveis a uma ou duas semanas atípicas.

As respostas são criadas __dentro__ de cada conjunto. Assim, as últimas 6 linhas de cada conjunto (cuja resposta cairia no conjunto seguinte) são descartadas, e nenhum valor de validação ou teste é usado como resposta no treino.

In [ ]:
n = len(df)
train_end = df.index[int(n * TRAIN_FRAC)].normalize()
val_end = df.index[int(n * (TRAIN_FRAC + VAL_FRAC))].normalize()


def make_targets(part):
    part = part.copy()
    for name, steps in HORIZONS.items():
        part[name] = part['appliances'].shift(-steps)
    return part.dropna(subset=list(HORIZONS)).astype({name: int for name in HORIZONS})


splits = {
    'train': make_targets(df[df.index < train_end]),
    'val': make_targets(df[(df.index >= train_end) & (df.index < val_end)]),
    'test': make_targets(df[df.index >= val_end]),
}

pd.DataFrame({
    name: {
        'início': s.index.min(),
        'fim': s.index.max(),
        'dias': (s.index.max() - s.index.min()).ceil('D').days,
        'n': len(s),
        'fração': round(len(s) / sum(len(p) for p in splits.values()), 3),
    }
    for name, s in splits.items()
}).T

In [ ]:
daily_split = pd.concat({name: s['appliances'].resample('D').mean() for name, s in splits.items()},
                        names=['conjunto', 'date']).reset_index()

fig = px.line(daily_split, x='date', y='appliances', color='conjunto',
              title='Consumo médio diário dos eletrodomésticos por conjunto',
              labels={'appliances': 'Wh', 'date': ''})
fig.show()

### Padronização

Padronizamos as variáveis contínuas com o `StandardScaler` do scikit-learn, ajustado __somente no treino__; usar estatísticas de validação ou teste seria vazamento de informação. As variáveis cíclicas e one-hot passam sem alteração (`remainder='passthrough'`).

As respostas `y_10` e `y_60` são a mesma variável que `appliances`, então usam um `StandardScaler` próprio ajustado sobre `appliances` no treino. Para reportar MAE, RMSE e MASE em Wh, as previsões devem ser revertidas com `unscale_y`.

O resultado são arrays numpy em `float32` (`X[conjunto]` e `y[conjunto]`), prontos para virar tensores do PyTorch com `torch.from_numpy`. As colunas de `X` seguem a ordem de `FEATURE_NAMES`, e as datas de cada linha continuam disponíveis em `splits[conjunto].index` (útil para avaliar o desempenho por hora do dia).

Obs.: como `appliances` é bastante assimétrica, um `log1p` antes da padronização pode ajudar no treino das redes. Por ora mantemos a escala original, para que todos partam da mesma base.

In [ ]:
FEATURES = df.columns.tolist()
TARGETS = list(HORIZONS)
SCALE_COLS = [c for c in FEATURES if not c.startswith(('hour_', 'dow_'))]

x_scaler = ColumnTransformer([('std', StandardScaler(), SCALE_COLS)], remainder='passthrough',
                             verbose_feature_names_out=False)
x_scaler.fit(splits['train'][FEATURES])
FEATURE_NAMES = x_scaler.get_feature_names_out().tolist()

y_scaler = StandardScaler().fit(splits['train'][['appliances']].to_numpy())


def scale_y(values):
    """Padroniza respostas em Wh de formato (n, k) com as estatísticas de `appliances`."""
    return y_scaler.transform(values.reshape(-1, 1)).reshape(values.shape)


def unscale_y(values):
    """Reverte previsões padronizadas para Wh."""
    return y_scaler.inverse_transform(values.reshape(-1, 1)).reshape(values.shape)


X = {name: x_scaler.transform(s[FEATURES]).astype(np.float32) for name, s in splits.items()}
y = {name: scale_y(s[TARGETS].to_numpy()).astype(np.float32) for name, s in splits.items()}

check_cols = ['appliances', 'kitchen_temp', 'weather_temp']
pd.concat({name: pd.DataFrame(X[name], columns=FEATURE_NAMES)[check_cols].agg(['mean', 'std'])
           for name in X}).round(2)

Como esperado, o treino tem média 0 e desvio 1, enquanto validação e teste se afastam disso: o período de teste (maio) é mais quente que o de treino (janeiro a abril). Os modelos terão que extrapolar para temperaturas pouco vistas no treino, algo a lembrar na seção "Quando o modelo de aprendizagem profunda falha?".

### Referência: previsão por persistência

A previsão por persistência ($\hat{y}_{t+h} = y_t$) é a referência do MASE. Definimos

$$\text{MASE} = \frac{\text{MAE}_{\text{modelo}}}{\text{MAE}_{\text{persistência, treino}}},$$

calculando o denominador no conjunto de treino e para o mesmo horizonte do modelo. Um MASE < 1 indica que o modelo supera a persistência.

In [ ]:
def persistence_metrics(part):
    rows = {}
    for name in HORIZONS:
        err = part[name] - part['appliances']
        rows[name] = {'MAE': err.abs().mean(), 'RMSE': np.sqrt((err ** 2).mean())}
    return pd.DataFrame(rows).T


MASE_SCALE = persistence_metrics(splits['train'])['MAE']

pd.concat({name: persistence_metrics(s) for name, s in splits.items()}, names=['conjunto', 'horizonte']).round(2)

__Próximos passos.__ Como todas as etapas acima são determinísticas e baratas, os conjuntos não são salvos em disco: basta rodar esta seção para obter exatamente os mesmos `X` e `y`. Os modelos sequenciais (RNN e CNN 1D) recebem janelas com os últimos $L$ passos, e os modelos AR e MLP recebem defasagens (lags) de `appliances`. Em ambos os casos, as janelas e lags devem ser construídos dentro de cada conjunto, e as primeiras $L$ linhas de cada conjunto ficam sem histórico completo.